# Lab 02 — Simulação e resposta temporal; identificação pelo degrau

**Unidade I — Modelagem e análise de sistemas físicos** · conteúdos 1.1/1.2 do PPC

**Objetivos:**
1. Dominar as funções de simulação temporal (`step_response`, `forced_response`, `initial_response`);
2. Extrair parâmetros de modelo a partir de dados "experimentais" com ruído;
3. Ajustar um modelo FOPDT (1ª ordem + tempo morto) pelo **método dos dois pontos**;
4. Validar o modelo identificado contra os dados.

**Referências:** Åström & Murray (FBS), caps. 5–6 · Ogata, cap. 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

rng = np.random.default_rng(42)  # gerador de números aleatórios com semente fixa

## 1. Resposta a entradas arbitrárias com `forced_response`

Além do degrau, plantas reais recebem rampas, pulsos e sinais compostos.
Vamos usar o motor CC reduzido do Lab 01: $G(s) = \dfrac{5}{2s+1}$ (valores arredondados).

In [ ]:
G = ct.tf([5], [2, 1])   # planta de referência desta aula

# entrada composta: degrau + rampa + pulso
t = np.linspace(0, 30, 3000)
u = np.zeros_like(t)
u[(t >= 1)] = 1.0                        # degrau em t = 1 s
u[(t >= 10)] += 0.1 * (t[(t >= 10)] - 10)  # rampa a partir de t = 10 s
u[(t >= 20) & (t < 22)] += 2.0           # pulso de 2 s em t = 20 s

resp = ct.forced_response(G, t, u)

fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axs[0].plot(t, u, 'C1', lw=2)
axs[0].set_ylabel('Entrada u(t)')
axs[0].grid(True)
axs[1].plot(resp.time, resp.outputs, 'C0', lw=2)
axs[1].set_ylabel('Saída y(t)')
axs[1].set_xlabel('Tempo [s]')
axs[1].grid(True)
fig.suptitle('Resposta a entrada arbitrária (forced_response)')
plt.show()

**Observe:** ao degrau o sistema converge para $Ku$; à rampa ele segue com atraso e erro
constante; o pulso curto mal é "sentido" (o sistema filtra sinais mais rápidos que $\tau$).

## 2. Resposta a condições iniciais

In [ ]:
# sistema de 2ª ordem subamortecido partindo de condição inicial não nula
G2 = ct.tf([25], [1, 2, 25])
sys2 = ct.ss(G2)                       # converte para espaço de estados
x0 = [0.5, 0.0]                        # estado inicial

resp0 = ct.initial_response(sys2, T=np.linspace(0, 6, 600), X0=x0)

plt.figure(figsize=(8, 4))
plt.plot(resp0.time, resp0.outputs, lw=2)
plt.xlabel('Tempo [s]')
plt.ylabel('y(t)')
plt.title('Resposta livre a condição inicial (entrada nula)')
plt.grid(True)
plt.show()

## 3. "Experimento" de identificação: degrau com ruído de medição

Vamos gerar dados sintéticos que imitam um ensaio de bancada: a planta "verdadeira" é
**desconhecida** para o aluno — um sistema com tempo morto:
$$G_{real}(s) = \frac{K e^{-\theta s}}{\tau s + 1}$$
O objetivo é recuperar $K$, $\tau$ e $\theta$ **somente a partir dos dados medidos**.

In [ ]:
# ---- planta "verdadeira" (em bancada, este bloco seria o equipamento físico) ----
K_true, tau_true, theta_true = 3.0, 4.0, 1.5

# aproximação de Padé de 1ª ordem para o tempo morto e^{-theta s}
num_pade, den_pade = ct.pade(theta_true, 3)
G_real = ct.tf([K_true], [tau_true, 1]) * ct.tf(num_pade, den_pade)

# ensaio: degrau de amplitude A com ruído de medição
A_step = 2.0
t_exp = np.linspace(0, 30, 601)
u_exp = A_step * (t_exp >= 0)
resp_exp = ct.forced_response(G_real, t_exp, u_exp)
y_exp = resp_exp.outputs + rng.normal(0, 0.05, size=t_exp.shape)  # ruído sigma = 0.05

plt.figure(figsize=(9, 4))
plt.plot(t_exp, y_exp, '.', ms=3, alpha=0.6, label='dados medidos (com ruído)')
plt.xlabel('Tempo [s]')
plt.ylabel('y(t)')
plt.title('Ensaio de resposta ao degrau (A = 2)')
plt.legend()
plt.grid(True)
plt.show()

## 4. Método dos dois pontos (Smith): 28,3 % e 63,2 %

Para o modelo FOPDT, os instantes em que a resposta atinge 28,3 % e 63,2 % do valor final
fornecem:
$$\tau = 1{,}5\,(t_{63{,}2} - t_{28{,}3}) \qquad\qquad \theta = t_{63{,}2} - \tau$$
e o ganho é $K = \Delta y_\infty / \Delta u$.

In [ ]:
# 1) ganho: média do trecho final dividido pela amplitude do degrau
y_inf = y_exp[t_exp > 25].mean()
K_hat = y_inf / A_step

# 2) instantes de 28,3% e 63,2% (primeira travessia, com dados suavizados)
y_smooth = np.convolve(y_exp, np.ones(15) / 15, mode='same')  # média móvel simples
t28 = t_exp[np.argmax(y_smooth >= 0.283 * y_inf)]
t63 = t_exp[np.argmax(y_smooth >= 0.632 * y_inf)]

tau_hat = 1.5 * (t63 - t28)
theta_hat = t63 - tau_hat

print(f"Identificado: K = {K_hat:.3f} (real {K_true}), "
      f"tau = {tau_hat:.3f} s (real {tau_true}), "
      f"theta = {theta_hat:.3f} s (real {theta_true})")

## 5. Validação: modelo identificado × dados

In [ ]:
num_p, den_p = ct.pade(max(theta_hat, 1e-6), 3)
G_hat = ct.tf([K_hat], [tau_hat, 1]) * ct.tf(num_p, den_p)

resp_hat = ct.forced_response(G_hat, t_exp, u_exp)

plt.figure(figsize=(9, 4))
plt.plot(t_exp, y_exp, '.', ms=3, alpha=0.4, label='dados medidos')
plt.plot(resp_hat.time, resp_hat.outputs, 'r', lw=2, label='modelo identificado')
plt.xlabel('Tempo [s]')
plt.ylabel('y(t)')
plt.title('Validação do modelo FOPDT identificado')
plt.legend()
plt.grid(True)
plt.show()

# erro quadrático médio da validação
rmse = np.sqrt(np.mean((y_exp - resp_hat.outputs) ** 2))
print(f"RMSE de validação: {rmse:.4f}")

**Boas práticas de identificação (usar no projeto final):**
- Validar com um conjunto de dados **diferente** do usado no ajuste (por exemplo, degrau de outra amplitude);
- Repetir o ensaio e verificar repetibilidade (não-linearidades aparecem como parâmetros que "mudam" com a amplitude);
- Registrar as condições do ensaio (ponto de operação, amplitude, taxa de amostragem).

---
> **🖼️ Figuras de apoio nos livros:**
> - Ogata, **Figura 5.2** — curva de resposta exponencial ao degrau, com a tangente inicial e as marcas em T, 2T, 3T, 4T. Cap. 5, §5.2, **p. 148** (p. 159 do PDF).
> - Ogata, **Figura 8.2** — ensaio de resposta ao degrau unitário da planta (1º método de Ziegler–Nichols). Cap. 8, §8.2, **p. 523** (p. 534 do PDF).
> - Ogata, **Figura 8.3** — curva de resposta em forma de S, com a tangente no ponto de inflexão definindo o atraso L e a constante T. Cap. 8, §8.2, **p. 523** (p. 534 do PDF).

## Exercícios (relatório do Lab 02)

**E1.** Refaça a identificação usando somente a **regra dos 63,2 %** (sem tempo morto,
$\theta = 0$) e compare o RMSE com o método dos dois pontos. O que o tempo morto ignorado causa?

**E2.** Repita o ensaio com ruído maior (`sigma = 0.2`). Os parâmetros identificados degradam?
Qual etapa do método é mais sensível ao ruído?

**E3.** Aplique um degrau de amplitude $A = 4$ na planta "real" e valide o modelo identificado
com $A = 2$. Comente sobre linearidade.

**E4.** Simule a resposta do modelo identificado a uma **onda quadrada** de período 20 s usando
`forced_response` e preveja o comportamento antes de rodar. A previsão se confirmou?

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui